In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np

import string
from re import sub

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score
import time

In [ ]:
train = pd.read_csv('/content/drive/My Drive/HTC_CNN_dataset/WOS/wos_train_final.csv')
test = pd.read_csv('/content/drive/My Drive/HTC_CNN_dataset/WOS/wos_test_final.csv')

In [ ]:
train_data = train
test_data = test  

tfidf = TfidfVectorizer()

In [ ]:
X_train_text = train_data['text']  
y_train_l1 = train_data['l1'] 

start_time = time.time() 
model_l1 = make_pipeline(tfidf, LogisticRegression()) 
model_l1.fit(X_train_text, y_train_l1) 
end_time = time.time()

time_l1 = round(end_time - start_time)
print(f'Time for l1: {time_l1}') 
train_pred_l1 = model_l1.predict(X_train_text) 

accuracy_l1 = accuracy_score(y_train_l1, train_pred_l1)

print(f'训练集 l1 的准确率: {accuracy_l1:.4f}')

Time for l1: 50
训练集 l1 的准确率: 0.9048


In [ ]:
X_train_l2 = pd.DataFrame({'text': X_train_text, 'pred_l1': train_pred_l1}) 
y_train_l2 = train_data['l2']

start_time = time.time() 
model_l2 = make_pipeline(TfidfVectorizer(), LogisticRegression())
model_l2.fit(X_train_l2.apply(lambda row: ' '.join(row.values.astype(str)), axis=1), y_train_l2) 
end_time = time.time()

time_l2 = round(end_time - start_time)
print(f'Time for l2: {time_l2}') 

train_pred_l2 = model_l2.predict(X_train_l2.apply(lambda row: ' '.join(row.values.astype(str)), axis=1))

accuracy_l2 = accuracy_score(y_train_l2, train_pred_l2) 
print(f'训练集 l2 的准确率: {accuracy_l2:.4f}')

Time for l2: 601
训练集 l2 的准确率: 0.8341


In [ ]:
X_test_text = test_data['text']
y_test_l1 = test_data['l1'].values
y_test_l2 = test_data['l2'].values

In [ ]:
test_pred_l1 = model_l1.predict(X_test_text)

correct_l1 = test_pred_l1 == test_data['l1'] 
correct_l2 = [False] * len(test_data) 

In [ ]:

accuracy_l1_test = accuracy_score(y_test_l1, test_pred_l1) 
print(f'测试集 l1 的准确率: {accuracy_l1_test:.4f}') 

test_pred_l2 = model_l2.predict(X_test_text)  
accuracy_l2_test = accuracy_score(y_test_l2, test_pred_l2) 
print(f'测试集 l2 的准确率: {accuracy_l2_test:.4f}') 

测试集 l1 的准确率: 0.8273
测试集 l2 的准确率: 0.6650


In [ ]:
test_accuracy_l1 = accuracy_score(test_data['l1'], test_pred_l1) 
print(f'Consistency 测试集 l1 的准确率: {test_accuracy_l1:.4f}')

Consistency 测试集 l1 的准确率: 0.8273


In [ ]:
X_test_l2 = pd.DataFrame({'text': X_test_text, 'pred_l1': test_pred_l1})
correct_l2_indices = X_test_l2[correct_l1].index 
test_pred_l2 = [None] * len(test_data) 
if not correct_l2_indices.empty:
    X_test_l2_correct = X_test_l2.loc[correct_l2_indices] 
    test_pred_l2_correct = model_l2.predict(X_test_l2_correct.apply(lambda row: ' '.join(row.values.astype(str)), axis=1))  # 预测L2标签
    for idx, pred in zip(correct_l2_indices, test_pred_l2_correct):
        test_pred_l2[idx] = pred
        correct_l2[idx] = pred == test_data['l2'].iloc[idx] 

In [ ]:
test_accuracy_l2 = sum(correct_l2) / len(test_data)
print(f'Consistency 测试集 l2 的准确率: {test_accuracy_l2:.4f}')

Consistency 测试集 l2 的准确率: 0.6096
